In [ ]:
import numpy as np
import random as random
import time as time
from itertools import combinations, chain
import joblib
from joblib import Parallel, delayed
import pickle
import matplotlib.pyplot as plt
import math

def load(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

In [ ]:
#cat_game_[class] loads results for runs with basic dynamics (no invention, no forgetting)
#inv_forg_[class] loads results for runs with invention and forgetting

basic_1_4 = load('cat_game_4-1.pkl')
basic_2_3 = load('cat_game_3-2.pkl')
basic_2_2 = load('cat_game_2-2.pkl')
basic_1_3 = load('cat_game_3-1.pkl')
basic_1_2 = load('cat_game_1-2.pkl')

inv_forg_1_4 = load('inv_forg_4-1.pkl')
inv_forg_2_3 = load('inv_forg_3-2.pkl')
inv_forg_2_2 = load('inv_forg_2-2.pkl')
inv_forg_1_3 = load('inv_forg_3-1.pkl')

In [ ]:
pooling_succ_rates = [0.5, 0.75, 0.6, 0.8, 2/3]

def show_results(results, strat_class):

    pooling_succ_rate = pooling_succ_rates[strat_class]
    

    optimal_partitions_probs = []
    success_rate_cumulative = []
    success_rate_final_1000 = []
    partition_prob_maxes = []
    pooling = 0
    separating = 0

    for i in range(len(results)):

        if strat_class == 0:  # {2,2}
            optimal_partitions_probs.append(
                sum(results[i][0][k] for k in [3, 4, 13, 14])
            )

        elif strat_class == 1:  # {1,3}
            optimal_partitions_probs.append(
                sum(results[i][0][k] for k in [9, 10, 12, 13, 14])
            )

        elif strat_class == 2:  # {2,3}
            optimal_partitions_probs.append(
                sum(results[i][0][k] for k in
                    [9,10,12,13,14,45,46,49,50,51])
            )

        elif strat_class == 3:  # {1,4}
            optimal_partitions_probs.append(
                sum(results[i][0][k] for k in
                    [29,30,32,33,34,40,41,42,44,45,46,48,49,50,51])
            )
            
        elif strat_class == 4:  # {1,2}
            optimal_partitions_probs.append(
                sum(results[i][0][k] for k in
                    [3,4])
            )

        success_rate_cumulative.append(
            np.mean(results[i][3])
        )

        success_rate_final_1000.append(
            np.mean(results[i][3][-1000:])
        )

        partition_prob_maxes.append(
            np.max(results[i][0])
        )

        if abs(np.mean(success_rate_final_1000[i]) - pooling_succ_rate) < 0.025:
            pooling += 1
        if 1 - np.mean(success_rate_final_1000[i]) < 0.05:
            separating += 1

    print(
        'probability of optimal partition being used:',
        round(np.mean(optimal_partitions_probs), 3)
    )

    print(
        'mean cumulative success rate:',
        round(np.mean(success_rate_cumulative), 3)
    )

    print(
        'mean success rate over final 1000 plays:',
        round(np.mean(success_rate_final_1000), 3)
    )

    print(
        'mean maximum probability for a single partition:',
        round(np.mean(partition_prob_maxes), 3)
    )

    print(
    'proportion of runs close to separating equilibrium:',
    round((separating / len(results)), 3)
    )

    print(
    'proportion of runs close to pooling equilibrium:',
    round((pooling / len(results)), 3)
    )

    bins = np.arange(0, 1.05, 0.05)

    plt.hist(partition_prob_maxes, bins=bins)
    plt.xlabel("max partition choice prob., end of run")
    plt.ylabel("no. runs")
    plt.xlim(0, 1)
    plt.ylim(0, 80)

    plt.show()
    print(' ')
    print('***')
    print(' ')


In [ ]:
# shows results for each strategic class and learning treatment

# correspondence between strat_class parameter and strategic class:
#    0 --> {2,2}
#    1 --> {1,3}
#    2 --> {2,3}
#    3 --> {1,4}
#    4 --> {1,2}

show_results(basic_2_2, 0)
show_results(basic_1_3, 1)
show_results(basic_2_3, 2)
show_results(basic_1_4, 3)
show_results(basic_1_2, 4)

show_results(inv_forg_2_2, 0)
show_results(inv_forg_1_3, 1)
show_results(inv_forg_2_3, 2)
show_results(inv_forg_1_4, 3)
